# 第 12 章 アンサンブル学習

弱い学習器を集めて強い学習器を作ります。多数決（バギング）と逐次的な重み付け（AdaBoost）を比べます。

対応する記事: [第 12 章 アンサンブル学習（Polyglot Notebook（F#） の言語版）](../../../docs/article/grokking-machine-learning/fsharp/ch12.md)

実装本体: `apps/grokking-ml-fsharp/src/`

## セットアップ

実装本体（`../src/GrokkingMl/`）を `#load` で読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

VS Code の [Polyglot Notebooks 拡張](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) で開くか、Jupyter に .NET Interactive カーネルを登録して実行します。

```bash
dotnet tool install -g Microsoft.dotnet-interactive
dotnet interactive jupyter install
jupyter lab notebooks/
```

In [1]:
#load "../src/GrokkingMl/Ch09DecisionTrees.fs"
#load "../src/GrokkingMl/Ch12Ensembles.fs"

open GrokkingMl.Ch09DecisionTrees
open GrokkingMl.Ch12Ensembles

## 切り株 1 本では解けないデータ

1 次元上に 3 つの領域が並んでいます。深さ 1 の木（切り株）は **線を 1 本しか引けない** ので、3 領域は分けられません。

In [2]:
let points: Point list = [ for i in 1..8 -> [ float i; 1.0 ] ]
let labels = [ 1; 1; -1; -1; -1; -1; 1; 1 ]

let stump = buildTreeWith giniImpurity 1 1 points labels
printfn "%A" stump
printfn "切り株 1 本の正解率 %.2f" (treeAccuracy stump points labels)

Node ({ Feature = 0
        Threshold = 2.5 }, Leaf 1, Leaf -1)

切り株 1 本の正解率 

0.75

## バギングと AdaBoost を比べる

**同じ弱学習器を同じ本数使っても、束ね方で結果が分かれます。**

バギングが効かないのは、復元抽出でデータを揺らしても **どの標本でも「x = 2.5 で切る」のが最良** なので、10 本ともほぼ同じ木になるからです。同じ意見を 10 回聞いても結論は変わりません。

In [3]:
let forest = trainForestWith 10 1 giniImpurity 0 points labels
let boosted = trainAdaBoostWith 10 1 giniImpurity points labels

printfn "切り株 1 本            %.2f" (treeAccuracy stump points labels)
printfn "バギング（10 本）      %.2f" (forestAccuracy forest points labels)
printfn "AdaBoost（10 本）      %.2f" (boostAccuracy boosted points labels)

切り株 1 本            

0.75

バギング（10 本）      

0.75

AdaBoost（10 本）      

1.00

## AdaBoost が生む役割分担

**1 本目は左端、2 本目は右端に注目しました。** 1 本目が右端を外したので、その点の重みが上がり、2 本目が拾いに行ったのです。誰も指示していないのに役割分担が生まれます。

In [4]:
boosted.Learners
|> List.truncate 4
|> List.iteri (fun index learner ->
    printfn "ラウンド %d  発言権 %.4f" (index + 1) learner.Weight
    printfn "          %A" learner.Tree)

ラウンド 

1

  発言権 

0.5493

Node ({ Feature = 0
        Threshold = 2.5 }, Leaf 1, Leaf -1)

...

ラウンド 

4

  発言権 

0.7332

Node ({ Feature = 0
        Threshold = 2.5 }, Leaf 1, Leaf -1)

## 発言権の式

**誤り率 0.5 でちょうど 0 になります。** 当てずっぽうの意見は無視されます。0.5 より悪い学習器は「逆を言えば当たる」ので負の発言権になります。

In [5]:
printfn "%8s %10s" "誤り率" "発言権"

for error in [ 0.0; 0.05; 0.2; 0.4; 0.5; 0.7 ] do
    printfn "%8.2f %10.4f" error (learnerWeight error)

     誤り率

       発言権

    0.00

   11.5129

    0.05

...

    0.2027

    0.50

    0.0000

    0.70

   -0.4236

## 試してみる: ラウンド数を変える

In [6]:
for rounds in [ 1; 2; 3; 5; 10 ] do
    let m = trainAdaBoostWith rounds 1 giniImpurity points labels
    printfn "%3d ラウンド  学習器 %2d 本  正解率 %.2f" rounds (List.length m.Learners)
        (boostAccuracy m points labels)

  1

 ラウンド  学習器 

 1

 本  正解率 

0.75

...

 10

 ラウンド  学習器 

10

 本  正解率 

1.00